# Библиотеки

In [1]:
from datasets import load_dataset
import matplotlib.pyplot as plt

c:\Users\aaron\vkr_tsa\project\vkr_gen_model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
import json
import re

# Подбор наборов данных

## [Vikhrmodels/russian_physics](https://huggingface.co/datasets/Vikhrmodels/russian_physics)

<u>Кол-во строк:</u> 98

* задачи по физике на русском языке, преимущественно олимпиадного и углублённого школьного уровня
* ориентирован на reasoning (пошаговое решение)
* качественная формулировка ответов, подойдёт для того, чтобы моделька могла именно "думать"


## [Vikhrmodels/russian_math](https://huggingface.co/datasets/Vikhrmodels/russian_math)

<u>Кол-во строк:</u> 199

* задачи по математике разного уровеня
* ориентирован на reasoning (пошаговое решение)
* понятное объяснение решений

## [Vikhrmodels/sdamgia](https://huggingface.co/datasets/Vikhrmodels/sdamgia)

<u>Рвзмер:</u>
1) Split 1 'mathb' - *2.5k*
2) Split 2 'math'  - *7.2k*

_____

* большое кол-во задач
* структурированные вопросы
* реальные экзаменационные задания
* есть изображения

**ПРОБЛЕМЫ:**

- нет объяснений
- => только на таких данных модель не сможет научиться "размышлять самостоятельно"

___

для использования этого набора данных необходимо добавить инструкции, + лучше смешивать с reasoning-датасетами

## [lighteval/QazUNTv2](https://huggingface.co/datasets/lighteval/QazUNTv2/viewer/ru)

<u>Рвзмер:</u>
1) Subset 1 'en' - *854*
2) Subset 2 'ru'  - *850*          -----------------> то, что подходит под задачу

_____

* датасет математических задач, использовавшийся для оценки моделей
* ориентирован на проверку reasoning
* может использоваться и для обучения и для валидации 

## [evilfreelancer/MATH-500-Russian](https://huggingface.co/datasets/evilfreelancer/MATH-500-Russian)

<u>Кол-во строк:</u> 500

* сложные мат. задачи
* ориентирован на reasoning

## [ai-bond/ru-alpaca-math](https://huggingface.co/datasets/ai-bond/ru-alpaca-math)

<u>Рвзмер:</u>
1) Split 1 'train' - *9,23k*
2) Split 2 'test'  - *7.02k*

______

* Instruction-датасет в стиле Alpaca, ориентированный на математические задачи
* разнообразные формулировки заданий

**ПРОБЛЕМЫ:**

- менее строгий стиль ответов
- возможен шум?? (синтетика)
- хорошо подойдёт для дополнения, но не как основной источник обучения

## [s85io/math-logic-ru](https://huggingface.co/datasets/s85io/math-logic-ru)

<u>Рвзмер:</u>
1) Split 1 'train' - *3,96k*
2) Split 2 'validation'  - *441*

____

* датасет по логике и математическим рассуждениям
* задачи на логическое мышление, выводы, формальные рассуждения
* усиливает у модели способность рассуждать  

_____________________________

## Доп. :

* Нужно, чтобы у всех формул был ЕДИНЫЙ формат, иначе модель будет очень плохо  их генерировать *=> необходимо привести все формулы в датасете к нужному формату* 

**LaTeX** - стандартный формат для таких записей

* Нужно добавить/переделать инструкции (где-то их нет, где-то они возможно не очень качественны)

* Могут быть проблемы с токенизацией спец. символов

# Разбор датасетов

## Vikhrmodels/russian_physics:

In [74]:
fiz = pd.read_parquet('data\physic.parquet')
fiz

,task,solution,answer,class,grade
0,"Небольшое тело, подвешенное на твёрдом стержне...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",2,school,8
1,"Небольшое тело, подвешенное на твёрдом стержне...","Вследствие удлинения маятника при нагревании, ...",10.8,school,8
2,"Катер пересёк прямую реку шириной 90 м, всё вр...",Катер смещается относительно берега за счёт ск...,6,school,8
3,"У Васи есть четыре одинаковых динамометра, оди...",Показания исправных одинаковых динамометров до...,3,school,8
4,"Однородный кирпич, имеющий форму прямоугольног...","Пусть длины рёбер кирпича равны a, b и c. Тогд...",3.125,school,8
...,...,...,...,...,...
93,В теплоизолированном сосуде находится вода при...,Запишем уравнение теплового баланса после погр...,41.5,school,10
94,"Закреплённая пушка, установленная на горизонта...","Центр масс двух осколков ""полетит"" по той же п...",360,school,11
95,Ракета удаляется от поверхности Земли с постоя...,угол 𝛼 должен лежать в диапазоне от 60° (вклю...,7.5,school,11
96,Посередине длинной доски массой 𝑀 = 4 кг сидит...,"Так как длина доски много больше её толщины, у...",44,school,11


In [17]:
fiz.info()

<class 'pandas.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   task      98 non-null     str  
 1   solution  98 non-null     str  
 2   answer    97 non-null     str  
 3   class     98 non-null     str  
 4   grade     98 non-null     int64
dtypes: int64(1), str(4)
memory usage: 156.4 KB


In [75]:
fiz[fiz.isna().any(axis = 1)]

,task,solution,answer,class,grade
34,В герметичном сосуде при температуре 𝑡1 = 47 ∘...,Пар в конце процесса будет насыщенным: 𝑝2 = 1 ...,NaN,school,11


In [21]:
print(fiz.loc[34, 'task'])
print('___________________')
print(fiz.loc[34, 'solution'])

В герметичном сосуде при температуре 𝑡1 = 47 ∘C и давлении 𝑝1 = 16 кПа находится одинаковое число молей воздуха и водяного пара. Сосуд медленно охлаждают до температуры 𝑡2 = 7∘C. Чему равно давление 𝑝2 в сосуде при температуре 𝑡2 ?
___________________
Пар в конце процесса будет насыщенным: 𝑝2 = 1 кПа+ 7/8 * 𝑝1/2 = 8 кПа.


меняю NaN на ответ:

In [76]:
fiz.loc[34, 'answer'] = '8'

In [23]:
fiz.info()

<class 'pandas.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   task      98 non-null     str  
 1   solution  98 non-null     str  
 2   answer    98 non-null     str  
 3   class     98 non-null     str  
 4   grade     98 non-null     int64
dtypes: int64(1), str(4)
memory usage: 156.4 KB


In [26]:
fiz['class'].value_counts()

class
school    87
allrus     9
region     1
1тур       1
Name: count, dtype: int64

In [27]:
fiz['grade'].value_counts()

grade
9     26
8     25
11    24
10    23
Name: count, dtype: int64

In [77]:
fiz = fiz.drop(columns = ['grade', 'class'])

In [29]:
fiz

,task,solution,answer
0,"Небольшое тело, подвешенное на твёрдом стержне...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",2
1,"Небольшое тело, подвешенное на твёрдом стержне...","Вследствие удлинения маятника при нагревании, ...",10.8
2,"Катер пересёк прямую реку шириной 90 м, всё вр...",Катер смещается относительно берега за счёт ск...,6
3,"У Васи есть четыре одинаковых динамометра, оди...",Показания исправных одинаковых динамометров до...,3
4,"Однородный кирпич, имеющий форму прямоугольног...","Пусть длины рёбер кирпича равны a, b и c. Тогд...",3.125
...,...,...,...
93,В теплоизолированном сосуде находится вода при...,Запишем уравнение теплового баланса после погр...,41.5
94,"Закреплённая пушка, установленная на горизонта...","Центр масс двух осколков ""полетит"" по той же п...",360
95,Ракета удаляется от поверхности Земли с постоя...,угол 𝛼 должен лежать в диапазоне от 60° (вклю...,7.5
96,Посередине длинной доски массой 𝑀 = 4 кг сидит...,"Так как длина доски много больше её толщины, у...",44


In [78]:
fiz['tags'] = 'physic'
fiz

,task,solution,answer,tags
0,"Небольшое тело, подвешенное на твёрдом стержне...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",2,physic
1,"Небольшое тело, подвешенное на твёрдом стержне...","Вследствие удлинения маятника при нагревании, ...",10.8,physic
2,"Катер пересёк прямую реку шириной 90 м, всё вр...",Катер смещается относительно берега за счёт ск...,6,physic
3,"У Васи есть четыре одинаковых динамометра, оди...",Показания исправных одинаковых динамометров до...,3,physic
4,"Однородный кирпич, имеющий форму прямоугольног...","Пусть длины рёбер кирпича равны a, b и c. Тогд...",3.125,physic
...,...,...,...,...
93,В теплоизолированном сосуде находится вода при...,Запишем уравнение теплового баланса после погр...,41.5,physic
94,"Закреплённая пушка, установленная на горизонта...","Центр масс двух осколков ""полетит"" по той же п...",360,physic
95,Ракета удаляется от поверхности Земли с постоя...,угол 𝛼 должен лежать в диапазоне от 60° (вклю...,7.5,physic
96,Посередине длинной доски массой 𝑀 = 4 кг сидит...,"Так как длина доски много больше её толщины, у...",44,physic


## Vikhrmodels/russian_math

In [69]:
math = pd.read_parquet('data\math.parquet')
math

,task,solution,short answer,class,grade
0,"Девять действительных a1, a2, ..., a9 образуют...","Пусть 𝑎 — первый член прогрессии, а 𝑑 — её раз...",-12,school,11
1,"В понедельник у Семёна был день рождения, ему ...","Обозначим через x количество рублей, которое п...",480,school,8
2,Учитель написал на доске четыре различных целы...,"Заметим, что число 37 является простым, и полу...",–111,school,8
3,За год каждый из восьмиклассников гимназии № 1...,"Для начала поймём, какие числа могут быть сред...",49,school,8
4,"По кругу стоят 36 детей, каждый из них одет в ...",Заметим. что не найдётся 3 стоящих подряд дево...,24,school,8
...,...,...,...,...,...
194,За круглым столом сидят 30 человек — рыцари и ...,"Из условия следует, что все сидящие за столом ...",0,region,9
195,На окружности отмечено 2N точек ( N — натураль...,Приведём другое доказательство шага индукции. ...,1,region,10
196,Петя выбрал натуральное число a > 1 и выписал ...,"Покажем сначала, что искомых чисел не может бы...",4,region,10
197,"2011 складов соединены дорогами так, что от лю...","Покажем вначале, что за 2009 рейсов план выпол...",2010,region,11


In [32]:
math.info()

<class 'pandas.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   task          199 non-null    str  
 1   solution      199 non-null    str  
 2   short answer  199 non-null    str  
 3   class         199 non-null    str  
 4   grade         199 non-null    int64
dtypes: int64(1), str(4)
memory usage: 295.5 KB


In [33]:
math['class'].value_counts()

class
school    95
msu       38
region    20
mipt      20
hse       13
ommo       9
allrus     4
Name: count, dtype: int64

In [34]:
math['grade'].value_counts()

grade
11    78
9     49
10    37
8     35
Name: count, dtype: int64

In [70]:
math = math.drop(columns = ['class', 'grade'])
math

,task,solution,short answer
0,"Девять действительных a1, a2, ..., a9 образуют...","Пусть 𝑎 — первый член прогрессии, а 𝑑 — её раз...",-12
1,"В понедельник у Семёна был день рождения, ему ...","Обозначим через x количество рублей, которое п...",480
2,Учитель написал на доске четыре различных целы...,"Заметим, что число 37 является простым, и полу...",–111
3,За год каждый из восьмиклассников гимназии № 1...,"Для начала поймём, какие числа могут быть сред...",49
4,"По кругу стоят 36 детей, каждый из них одет в ...",Заметим. что не найдётся 3 стоящих подряд дево...,24
...,...,...,...
194,За круглым столом сидят 30 человек — рыцари и ...,"Из условия следует, что все сидящие за столом ...",0
195,На окружности отмечено 2N точек ( N — натураль...,Приведём другое доказательство шага индукции. ...,1
196,Петя выбрал натуральное число a > 1 и выписал ...,"Покажем сначала, что искомых чисел не может бы...",4
197,"2011 складов соединены дорогами так, что от лю...","Покажем вначале, что за 2009 рейсов план выпол...",2010


In [71]:
math.rename(columns = {'short answer': 'answer'}, inplace = True)
math

,task,solution,answer
0,"Девять действительных a1, a2, ..., a9 образуют...","Пусть 𝑎 — первый член прогрессии, а 𝑑 — её раз...",-12
1,"В понедельник у Семёна был день рождения, ему ...","Обозначим через x количество рублей, которое п...",480
2,Учитель написал на доске четыре различных целы...,"Заметим, что число 37 является простым, и полу...",–111
3,За год каждый из восьмиклассников гимназии № 1...,"Для начала поймём, какие числа могут быть сред...",49
4,"По кругу стоят 36 детей, каждый из них одет в ...",Заметим. что не найдётся 3 стоящих подряд дево...,24
...,...,...,...
194,За круглым столом сидят 30 человек — рыцари и ...,"Из условия следует, что все сидящие за столом ...",0
195,На окружности отмечено 2N точек ( N — натураль...,Приведём другое доказательство шага индукции. ...,1
196,Петя выбрал натуральное число a > 1 и выписал ...,"Покажем сначала, что искомых чисел не может бы...",4
197,"2011 складов соединены дорогами так, что от лю...","Покажем вначале, что за 2009 рейсов план выпол...",2010


In [72]:
math['tags'] = 'math'
math

,task,solution,answer,tags
0,"Девять действительных a1, a2, ..., a9 образуют...","Пусть 𝑎 — первый член прогрессии, а 𝑑 — её раз...",-12,math
1,"В понедельник у Семёна был день рождения, ему ...","Обозначим через x количество рублей, которое п...",480,math
2,Учитель написал на доске четыре различных целы...,"Заметим, что число 37 является простым, и полу...",–111,math
3,За год каждый из восьмиклассников гимназии № 1...,"Для начала поймём, какие числа могут быть сред...",49,math
4,"По кругу стоят 36 детей, каждый из них одет в ...",Заметим. что не найдётся 3 стоящих подряд дево...,24,math
...,...,...,...,...
194,За круглым столом сидят 30 человек — рыцари и ...,"Из условия следует, что все сидящие за столом ...",0,math
195,На окружности отмечено 2N точек ( N — натураль...,Приведём другое доказательство шага индукции. ...,1,math
196,Петя выбрал натуральное число a > 1 и выписал ...,"Покажем сначала, что искомых чисел не может бы...",4,math
197,"2011 складов соединены дорогами так, что от лю...","Покажем вначале, что за 2009 рейсов план выпол...",2010,math


In [79]:
vikhr = pd.concat([fiz, math], ignore_index = True)
vikhr

,task,solution,answer,tags
0,"Небольшое тело, подвешенное на твёрдом стержне...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",2,physic
1,"Небольшое тело, подвешенное на твёрдом стержне...","Вследствие удлинения маятника при нагревании, ...",10.8,physic
2,"Катер пересёк прямую реку шириной 90 м, всё вр...",Катер смещается относительно берега за счёт ск...,6,physic
3,"У Васи есть четыре одинаковых динамометра, оди...",Показания исправных одинаковых динамометров до...,3,physic
4,"Однородный кирпич, имеющий форму прямоугольног...","Пусть длины рёбер кирпича равны a, b и c. Тогд...",3.125,physic
...,...,...,...,...
292,За круглым столом сидят 30 человек — рыцари и ...,"Из условия следует, что все сидящие за столом ...",0,math
293,На окружности отмечено 2N точек ( N — натураль...,Приведём другое доказательство шага индукции. ...,1,math
294,Петя выбрал натуральное число a > 1 и выписал ...,"Покажем сначала, что искомых чисел не может бы...",4,math
295,"2011 складов соединены дорогами так, что от лю...","Покажем вначале, что за 2009 рейсов план выпол...",2010,math


In [39]:
vikhr.info()

<class 'pandas.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   task      297 non-null    str  
 1   solution  297 non-null    str  
 2   answer    297 non-null    str  
 3   tags      297 non-null    str  
dtypes: str(4)
memory usage: 449.4 KB


In [40]:
vikhr.describe()

,task,solution,answer,tags
count,297,297,297,297
unique,294,297,184,2
top,С каким и в какую сторону направленным ускорен...,"Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",4,math
freq,2,1,13,199


In [80]:
vikhr = vikhr.drop_duplicates()

In [43]:
vikhr

,task,solution,answer,tags
0,"Небольшое тело, подвешенное на твёрдом стержне...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",2,physic
1,"Небольшое тело, подвешенное на твёрдом стержне...","Вследствие удлинения маятника при нагревании, ...",10.8,physic
2,"Катер пересёк прямую реку шириной 90 м, всё вр...",Катер смещается относительно берега за счёт ск...,6,physic
3,"У Васи есть четыре одинаковых динамометра, оди...",Показания исправных одинаковых динамометров до...,3,physic
4,"Однородный кирпич, имеющий форму прямоугольног...","Пусть длины рёбер кирпича равны a, b и c. Тогд...",3.125,physic
...,...,...,...,...
292,За круглым столом сидят 30 человек — рыцари и ...,"Из условия следует, что все сидящие за столом ...",0,math
293,На окружности отмечено 2N точек ( N — натураль...,Приведём другое доказательство шага индукции. ...,1,math
294,Петя выбрал натуральное число a > 1 и выписал ...,"Покажем сначала, что искомых чисел не может бы...",4,math
295,"2011 складов соединены дорогами так, что от лю...","Покажем вначале, что за 2009 рейсов план выпол...",2010,math


In [44]:
print(vikhr['task'].describe()['top'])

С каким и в какую сторону направленным ускорением нужно двигать средний блок, чтобы левый груз, имеющий массу 3 кг, оставался неподвижным? Массой нити и блоков можно пренебречь. Нить нерастяжима, трение отсутствует. g = 10 м/с**2 .


In [81]:
top = vikhr['task'].describe()['top']
vikhr[vikhr['task'] == top]

,task,solution,answer,tags
57,С каким и в какую сторону направленным ускорен...,Пусть m – масса правого груза. Чтобы левый гру...,10,physic
60,С каким и в какую сторону направленным ускорен...,Пусть m – масса правого груза. Чтобы левый гру...,2.5,physic


In [82]:
print(vikhr.loc[57, 'task'])
print('_______')
print(vikhr.loc[60, 'solution'])

С каким и в какую сторону направленным ускорением нужно двигать средний блок, чтобы левый груз, имеющий массу 3 кг, оставался неподвижным? Массой нити и блоков можно пренебречь. Нить нерастяжима, трение отсутствует. g = 10 м/с**2 .
_______
Пусть m – масса правого груза. Чтобы левый груз оставался в покое, натяжение нити должно равняться 𝑇 = 3/2𝑚𝑔. Тогда правый груз будет двигаться вверх с ускорением 𝑎1 = (3/2𝑚𝑔−𝑚𝑔)/𝑚 = 1/2𝑔. Поскольку левый конец нити неподвижен, средний блок должен двигаться вниз с ускорением 𝑎 = 1/4𝑔 = 2.5 м/с**2


In [83]:
vikhr = vikhr.drop(57)

In [84]:
vikhr.describe()

,task,solution,answer,tags
count,296,296,296,296
unique,294,296,184,2
top,"Девять действительных a1, a2, ..., a9 образуют...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",4,math
freq,2,1,13,199


In [85]:
top = vikhr['task'].describe()['top']
vikhr[vikhr['task'] == top]

,task,solution,answer,tags
98,"Девять действительных a1, a2, ..., a9 образуют...","Пусть 𝑎 — первый член прогрессии, а 𝑑 — её раз...",-12,math
175,"Девять действительных a1, a2, ..., a9 образуют...","Пусть 𝑎 — первый член прогрессии, а 𝑑 — её раз...",-12,math


In [86]:
vikhr = vikhr.drop(175)

In [87]:
vikhr.describe()

,task,solution,answer,tags
count,295,295,295,295
unique,294,295,184,2
top,Каждый из 10 гномов либо всегда говорит правду...,"Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",4,math
freq,2,1,13,198


In [88]:
top = vikhr['task'].describe()['top']
vikhr[vikhr['task'] == top]

,task,solution,answer,tags
168,Каждый из 10 гномов либо всегда говорит правду...,"Гномы, которые всегда говорят правду, подняли ...",4,math
286,Каждый из 10 гномов либо всегда говорит правду...,"Гномы, которые всегда говорят правду, подняли ...",4,math


In [89]:
print(vikhr.loc[168, 'solution'])
print('_______')
print(vikhr.loc[286, 'solution'])

Гномы, которые всегда говорят правду, подняли руку один раз, а гномы, которые всегда лгут, – два раза. Всего было поднято 16 рук (10 + 5 + 1). Если бы все гномы сказали правду, то было бы поднято 10 рук. Если одного правдивого гнома заменить на одного лгуна, то число поднятых рук увеличится на 1. Так как было поднято 6 «лишних» рук, то 6 гномов солгали, а 4 сказали правду
_______
Гномы, которые всегда говорят правду, подняли руку один раз, а гномы, которые всегда лгут, – два раза. Всего было поднято 16 рук (10 + 5 + 1). Если бы все гномы сказали правду, то было бы поднято 10 рук. Если одного правдивого гнома заменить на одного лгуна, то число поднятых рук увеличится на 1. Так как было поднято 6 «лишних» рук, то 6 гномов солгали, а 4 сказали правду.


In [90]:
vikhr = vikhr.drop(286)

In [91]:
vikhr.describe()

,task,solution,answer,tags
count,294,294,294,294
unique,294,294,184,2
top,"Небольшое тело, подвешенное на твёрдом стержне...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",4,math
freq,1,1,12,197


In [92]:
top = vikhr['task'].describe()['top']
vikhr[vikhr['task'] == top]

,task,solution,answer,tags
0,"Небольшое тело, подвешенное на твёрдом стержне...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",2,physic


=> повторных задач не осталось

In [75]:
vikhr.to_json('data/vikhr.json', lines=True, force_ascii=False, orient = 'records')

## Vikhrmodels/physics_big

In [17]:
fiz_big = load_dataset('Vikhrmodels/physics_big')

In [18]:
fiz_big 

DatasetDict({
    train: Dataset({
        features: ['images', 'text', 'answer', 'tags'],
        num_rows: 2332
    })
})

In [29]:
fiz_df = fiz_big['train'].to_pandas()
fiz_df

,images,text,answer,tags
0,[],"{""text"": [{""text"": ""В космосе летает мыльный п...","[{""condition"": ""Найдите электрический заряд $q...","[""T"", ""Жаутыковские""]"
1,[],"{""text"": [{""text"": ""Испытывается новый скорост...","[{""condition"": ""Считая, что пули застревают в ...","[""T"", ""Жаутыковские""]"
2,[],"{""text"": [{""text"": ""Фотографировать тигра с ра...","[{""condition"": ""Какой размер может иметь камер...","[""T"", ""Оптика"", ""Геометрическая оптика"", ""Всер..."
3,[],"{""text"": [{""text"": ""В калориметр наливают ложк...","[{""condition"": ""На сколько еще градусов возрас...","[""T"", ""Всероссийские"", ""Теплота"", ""Температура""]"
4,[],"{""text"": [{""text"": ""С одним молем идеального о...","[{""condition"": ""Найдите полную работу $A$ газа...","[""T"", ""Жаутыковские""]"
...,...,...,...,...
2327,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""Оборудование (оптическое) ...","[{""condition"": ""Получившийся спектр сохраните ...","[""E"", ""Оптика"", ""Интенсивность"", ""Спектр"", ""По..."
2328,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""Электролитами называют вещ...","[{""condition"": ""Измерьте внутренний диаметр шп...","[""E"", ""Цепи"", ""Емкость"", ""Сборы XY"", ""X"", ""Пер..."
2329,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""При разработке полупроводн...","[{""condition"": ""Измерение методом 4РР. Измерьт...","[""E"", ""Международные""]"
2330,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""<img_0> Во всей работе сиг...","[{""condition"": ""Измерьте сопротивления катушек...","[""E"", ""Цепи"", ""Осциллограф"", ""Индуктивность"", ..."


In [30]:
def safe_parse(x):
    if not isinstance(x, str):
        return x
    
    try:
        return json.loads(x)
    except json.JSONDecodeError:
        # фиксим одинарные слеши → двойные
        fixed = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', x)
        
        try:
            return json.loads(fixed)
        except:
            return None  # или можно вернуть исходную строку

fiz_df['parsed'] = fiz_df['text'].apply(safe_parse)

In [31]:
fiz_df

,images,text,answer,tags,parsed
0,[],"{""text"": [{""text"": ""В космосе летает мыльный п...","[{""condition"": ""Найдите электрический заряд $q...","[""T"", ""Жаутыковские""]",{'text': [{'text': 'В космосе летает мыльный п...
1,[],"{""text"": [{""text"": ""Испытывается новый скорост...","[{""condition"": ""Считая, что пули застревают в ...","[""T"", ""Жаутыковские""]",{'text': [{'text': 'Испытывается новый скорост...
2,[],"{""text"": [{""text"": ""Фотографировать тигра с ра...","[{""condition"": ""Какой размер может иметь камер...","[""T"", ""Оптика"", ""Геометрическая оптика"", ""Всер...",{'text': [{'text': 'Фотографировать тигра с ра...
3,[],"{""text"": [{""text"": ""В калориметр наливают ложк...","[{""condition"": ""На сколько еще градусов возрас...","[""T"", ""Всероссийские"", ""Теплота"", ""Температура""]",{'text': [{'text': 'В калориметр наливают ложк...
4,[],"{""text"": [{""text"": ""С одним молем идеального о...","[{""condition"": ""Найдите полную работу $A$ газа...","[""T"", ""Жаутыковские""]",{'text': [{'text': 'С одним молем идеального о...
...,...,...,...,...,...
2327,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""Оборудование (оптическое) ...","[{""condition"": ""Получившийся спектр сохраните ...","[""E"", ""Оптика"", ""Интенсивность"", ""Спектр"", ""По...",None
2328,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""Электролитами называют вещ...","[{""condition"": ""Измерьте внутренний диаметр шп...","[""E"", ""Цепи"", ""Емкость"", ""Сборы XY"", ""X"", ""Пер...",{'text': [{'text': 'Электролитами называют вещ...
2329,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""При разработке полупроводн...","[{""condition"": ""Измерение методом 4РР. Измерьт...","[""E"", ""Международные""]",{'text': [{'text': 'При разработке полупроводн...
2330,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""<img_0> Во всей работе сиг...","[{""condition"": ""Измерьте сопротивления катушек...","[""E"", ""Цепи"", ""Осциллограф"", ""Индуктивность"", ...",{'text': [{'text': '<img_0> Во всей работе сиг...


In [32]:
type(fiz_df['parsed'].iloc[0])

dict

In [35]:
fiz_df['parsed'].iloc[0]['text']

[{'text': 'В космосе летает мыльный пузырь радиуса $R_1$. С помощью внешнего ионизатора мыльную пленку быстро заряжают некоторым положительным зарядом, после чего радиус пузыря через некоторое время перестаёт меняться и становится равным $R_2=2R_1$.'},
 {'balls': {'number': '1.', 'exponent': '4,00'},
  'text': 'Найдите электрический заряд $q$, который был сообщен мыльной пленке, если ее теплоемкость и теплопроводность ничтожно малы. Коэффициент поверхностного натяжения мыльной пленки не зависит от температуры и равен $\\sigma$. Воздух считать идеальным двухатомным газом.'}]

In [38]:
fiz_df['parsed'].iloc[0]['text'][1]

{'balls': {'number': '1.', 'exponent': '4,00'},
 'text': 'Найдите электрический заряд $q$, который был сообщен мыльной пленке, если ее теплоемкость и теплопроводность ничтожно малы. Коэффициент поверхностного натяжения мыльной пленки не зависит от температуры и равен $\\sigma$. Воздух считать идеальным двухатомным газом.'}

In [39]:
def extract_question(x):
    try:
        return x['text'][0]['text']
    except:
        return None

fiz_df['question'] = fiz_df['parsed'].apply(extract_question)

In [41]:
fiz_df

,images,text,answer,tags,parsed,question
0,[],"{""text"": [{""text"": ""В космосе летает мыльный п...","[{""condition"": ""Найдите электрический заряд $q...","[""T"", ""Жаутыковские""]",{'text': [{'text': 'В космосе летает мыльный п...,В космосе летает мыльный пузырь радиуса $R_1$....
1,[],"{""text"": [{""text"": ""Испытывается новый скорост...","[{""condition"": ""Считая, что пули застревают в ...","[""T"", ""Жаутыковские""]",{'text': [{'text': 'Испытывается новый скорост...,Испытывается новый скорострельный многоствольн...
2,[],"{""text"": [{""text"": ""Фотографировать тигра с ра...","[{""condition"": ""Какой размер может иметь камер...","[""T"", ""Оптика"", ""Геометрическая оптика"", ""Всер...",{'text': [{'text': 'Фотографировать тигра с ра...,Фотографировать тигра с расстояния менее $20~м...
3,[],"{""text"": [{""text"": ""В калориметр наливают ложк...","[{""condition"": ""На сколько еще градусов возрас...","[""T"", ""Всероссийские"", ""Теплота"", ""Температура""]",{'text': [{'text': 'В калориметр наливают ложк...,"В калориметр наливают ложку горячей воды, при ..."
4,[],"{""text"": [{""text"": ""С одним молем идеального о...","[{""condition"": ""Найдите полную работу $A$ газа...","[""T"", ""Жаутыковские""]",{'text': [{'text': 'С одним молем идеального о...,С одним молем идеального одноатомного газа про...
...,...,...,...,...,...,...
2327,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""Оборудование (оптическое) ...","[{""condition"": ""Получившийся спектр сохраните ...","[""E"", ""Оптика"", ""Интенсивность"", ""Спектр"", ""По...",None,NaN
2328,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""Электролитами называют вещ...","[{""condition"": ""Измерьте внутренний диаметр шп...","[""E"", ""Цепи"", ""Емкость"", ""Сборы XY"", ""X"", ""Пер...",{'text': [{'text': 'Электролитами называют вещ...,"Электролитами называют вещества, которое прово..."
2329,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""При разработке полупроводн...","[{""condition"": ""Измерение методом 4РР. Измерьт...","[""E"", ""Международные""]",{'text': [{'text': 'При разработке полупроводн...,При разработке полупроводниковых устройств (ко...
2330,[{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIH...,"{""text"": [{""text"": ""<img_0> Во всей работе сиг...","[{""condition"": ""Измерьте сопротивления катушек...","[""E"", ""Цепи"", ""Осциллограф"", ""Индуктивность"", ...",{'text': [{'text': '<img_0> Во всей работе сиг...,<img_0> Во всей работе сигнал генератора синус...


In [43]:
fiz_df['question'].loc[0]

'В космосе летает мыльный пузырь радиуса $R_1$. С помощью внешнего ионизатора мыльную пленку быстро заряжают некоторым положительным зарядом, после чего радиус пузыря через некоторое время перестаёт меняться и становится равным $R_2=2R_1$.'

In [45]:
print(fiz_df['parsed'].iloc[0])

{'text': [{'text': 'В космосе летает мыльный пузырь радиуса $R_1$. С помощью внешнего ионизатора мыльную пленку быстро заряжают некоторым положительным зарядом, после чего радиус пузыря через некоторое время перестаёт меняться и становится равным $R_2=2R_1$.'}, {'balls': {'number': '1.', 'exponent': '4,00'}, 'text': 'Найдите электрический заряд $q$, который был сообщен мыльной пленке, если ее теплоемкость и теплопроводность ничтожно малы. Коэффициент поверхностного натяжения мыльной пленки не зависит от температуры и равен $\\sigma$. Воздух считать идеальным двухатомным газом.'}]}


## lighteval/QazUNTv2

In [46]:
qaz = load_dataset('lighteval/QazUNTv2', 'ru')

c:\Users\aaron\vkr_tsa\project\vkr_gen_model\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aaron\.cache\huggingface\hub\datasets--lighteval--QazUNTv2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 850/850 [00:00<00:00, 5394.72 examples/s]


In [47]:
qaz

DatasetDict({
    train: Dataset({
        features: ['id', 'section', 'question', 'solution', 'answer', 'options', '__index_level_0__'],
        num_rows: 850
    })
})

In [49]:
qaz_df = qaz['train'].to_pandas()
qaz_df

,id,section,question,solution,answer,options,__index_level_0__
0,1,algebra,Каким числом оканчавается выражение 9^121,"Начнем с того, что 9^1 = 9, 9^2 = 81, 9^3 = 72...",D,"[1, 7, 3, 9, 5]",0
1,2,logic,Какое число соответсвует вопросительному знаку...,Закономерность данного ряда следующая: к перво...,C,"[155, 75, 53, 99, 57]",1
2,3,logic,Какое число должно быть вместо вопросительного...,Для выполнения данного задания необходимо допи...,B,"[15, 25, 33, 45, 43]",2
3,4,algebra,"Среднее арифметическое шести чисел равно 70, а...","Пусть сумма шести чисел равна S1, а сумма четы...",B,"[85, 82, 17, 14, 36]",3
4,5,logic,"Замените буквы цифрами так, чтобы результат сл...","Слагаемые - числа четырёхзначные, а сумма - чи...",A,"[произведение различных цифр кратно 120, произ...",4
...,...,...,...,...,...,...,...
845,907,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых, 1 зеленый ...",D,"[4, 5, 6, 7, 8]",906
846,908,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых и 1 зеленый...",E,"[4, 5, 6, 7, 8]",907
847,909,logic,"В слове МАТЕМАТИКА стерли 6 букв (возможно, ср...",Перепишем слово в обратном порядке: АКИТАМЕТАМ...,D,"[КАМА, КИМА, АТЕМ, ТЕМА, ТАЕТ]",908
848,910,logic,"Восстановив порядок букв в каждой группе, вы п...","Если восстановить данные слова, получим: БАКУ,...",C,"[КУАБ, ТАНАСА, МАРСАКАНД, ВАКСОМ, ЕШКБИК]",909


In [50]:
qaz_df['section'].value_counts()

section
algebra        415
logic          296
probability    139
Name: count, dtype: int64

In [51]:
def get_real_answer(row):
    try:
        letter = row['answer']
        options = row['options']
        
        # если options вдруг строка → превращаем в список
        if isinstance(options, str):
            import ast
            options = ast.literal_eval(options)
        
        index = ord(letter) - ord('A')
        return options[index]
    
    except:
        return None

qaz_df['answer_text'] = qaz_df.apply(get_real_answer, axis=1)

In [52]:
qaz_df

,id,section,question,solution,answer,options,__index_level_0__,answer_text
0,1,algebra,Каким числом оканчавается выражение 9^121,"Начнем с того, что 9^1 = 9, 9^2 = 81, 9^3 = 72...",D,"[1, 7, 3, 9, 5]",0,9
1,2,logic,Какое число соответсвует вопросительному знаку...,Закономерность данного ряда следующая: к перво...,C,"[155, 75, 53, 99, 57]",1,53
2,3,logic,Какое число должно быть вместо вопросительного...,Для выполнения данного задания необходимо допи...,B,"[15, 25, 33, 45, 43]",2,25
3,4,algebra,"Среднее арифметическое шести чисел равно 70, а...","Пусть сумма шести чисел равна S1, а сумма четы...",B,"[85, 82, 17, 14, 36]",3,82
4,5,logic,"Замените буквы цифрами так, чтобы результат сл...","Слагаемые - числа четырёхзначные, а сумма - чи...",A,"[произведение различных цифр кратно 120, произ...",4,произведение различных цифр кратно 120
...,...,...,...,...,...,...,...,...
845,907,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых, 1 зеленый ...",D,"[4, 5, 6, 7, 8]",906,7
846,908,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых и 1 зеленый...",E,"[4, 5, 6, 7, 8]",907,8
847,909,logic,"В слове МАТЕМАТИКА стерли 6 букв (возможно, ср...",Перепишем слово в обратном порядке: АКИТАМЕТАМ...,D,"[КАМА, КИМА, АТЕМ, ТЕМА, ТАЕТ]",908,ТЕМА
848,910,logic,"Восстановив порядок букв в каждой группе, вы п...","Если восстановить данные слова, получим: БАКУ,...",C,"[КУАБ, ТАНАСА, МАРСАКАНД, ВАКСОМ, ЕШКБИК]",909,МАРСАКАНД


In [53]:
qaz_df['section'] = qaz_df['section'].replace('algebra', 'math')
qaz_df['section'] = qaz_df['section'].replace('probability', 'math')
qaz_df

,id,section,question,solution,answer,options,__index_level_0__,answer_text
0,1,math,Каким числом оканчавается выражение 9^121,"Начнем с того, что 9^1 = 9, 9^2 = 81, 9^3 = 72...",D,"[1, 7, 3, 9, 5]",0,9
1,2,logic,Какое число соответсвует вопросительному знаку...,Закономерность данного ряда следующая: к перво...,C,"[155, 75, 53, 99, 57]",1,53
2,3,logic,Какое число должно быть вместо вопросительного...,Для выполнения данного задания необходимо допи...,B,"[15, 25, 33, 45, 43]",2,25
3,4,math,"Среднее арифметическое шести чисел равно 70, а...","Пусть сумма шести чисел равна S1, а сумма четы...",B,"[85, 82, 17, 14, 36]",3,82
4,5,logic,"Замените буквы цифрами так, чтобы результат сл...","Слагаемые - числа четырёхзначные, а сумма - чи...",A,"[произведение различных цифр кратно 120, произ...",4,произведение различных цифр кратно 120
...,...,...,...,...,...,...,...,...
845,907,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых, 1 зеленый ...",D,"[4, 5, 6, 7, 8]",906,7
846,908,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых и 1 зеленый...",E,"[4, 5, 6, 7, 8]",907,8
847,909,logic,"В слове МАТЕМАТИКА стерли 6 букв (возможно, ср...",Перепишем слово в обратном порядке: АКИТАМЕТАМ...,D,"[КАМА, КИМА, АТЕМ, ТЕМА, ТАЕТ]",908,ТЕМА
848,910,logic,"Восстановив порядок букв в каждой группе, вы п...","Если восстановить данные слова, получим: БАКУ,...",C,"[КУАБ, ТАНАСА, МАРСАКАНД, ВАКСОМ, ЕШКБИК]",909,МАРСАКАНД


In [54]:
qaz_df = qaz_df.drop(columns = ['options', '__index_level_0__', 'id'])
qaz_df

,section,question,solution,answer,answer_text
0,math,Каким числом оканчавается выражение 9^121,"Начнем с того, что 9^1 = 9, 9^2 = 81, 9^3 = 72...",D,9
1,logic,Какое число соответсвует вопросительному знаку...,Закономерность данного ряда следующая: к перво...,C,53
2,logic,Какое число должно быть вместо вопросительного...,Для выполнения данного задания необходимо допи...,B,25
3,math,"Среднее арифметическое шести чисел равно 70, а...","Пусть сумма шести чисел равна S1, а сумма четы...",B,82
4,logic,"Замените буквы цифрами так, чтобы результат сл...","Слагаемые - числа четырёхзначные, а сумма - чи...",A,произведение различных цифр кратно 120
...,...,...,...,...,...
845,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых, 1 зеленый ...",D,7
846,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых и 1 зеленый...",E,8
847,logic,"В слове МАТЕМАТИКА стерли 6 букв (возможно, ср...",Перепишем слово в обратном порядке: АКИТАМЕТАМ...,D,ТЕМА
848,logic,"Восстановив порядок букв в каждой группе, вы п...","Если восстановить данные слова, получим: БАКУ,...",C,МАРСАКАНД


In [55]:
qaz_df = qaz_df.drop(columns = ['answer'])

In [56]:
qaz_df = qaz_df.rename(columns = {'answer_text': 'answer'})
qaz_df

,section,question,solution,answer
0,math,Каким числом оканчавается выражение 9^121,"Начнем с того, что 9^1 = 9, 9^2 = 81, 9^3 = 72...",9
1,logic,Какое число соответсвует вопросительному знаку...,Закономерность данного ряда следующая: к перво...,53
2,logic,Какое число должно быть вместо вопросительного...,Для выполнения данного задания необходимо допи...,25
3,math,"Среднее арифметическое шести чисел равно 70, а...","Пусть сумма шести чисел равна S1, а сумма четы...",82
4,logic,"Замените буквы цифрами так, чтобы результат сл...","Слагаемые - числа четырёхзначные, а сумма - чи...",произведение различных цифр кратно 120
...,...,...,...,...
845,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых, 1 зеленый ...",7
846,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых и 1 зеленый...",8
847,logic,"В слове МАТЕМАТИКА стерли 6 букв (возможно, ср...",Перепишем слово в обратном порядке: АКИТАМЕТАМ...,ТЕМА
848,logic,"Восстановив порядок букв в каждой группе, вы п...","Если восстановить данные слова, получим: БАКУ,...",МАРСАКАНД


In [57]:
qaz_df = qaz_df.rename(columns = {'question': 'task', 'section': 'tags'})
qaz_df

,tags,task,solution,answer
0,math,Каким числом оканчавается выражение 9^121,"Начнем с того, что 9^1 = 9, 9^2 = 81, 9^3 = 72...",9
1,logic,Какое число соответсвует вопросительному знаку...,Закономерность данного ряда следующая: к перво...,53
2,logic,Какое число должно быть вместо вопросительного...,Для выполнения данного задания необходимо допи...,25
3,math,"Среднее арифметическое шести чисел равно 70, а...","Пусть сумма шести чисел равна S1, а сумма четы...",82
4,logic,"Замените буквы цифрами так, чтобы результат сл...","Слагаемые - числа четырёхзначные, а сумма - чи...",произведение различных цифр кратно 120
...,...,...,...,...
845,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых, 1 зеленый ...",7
846,logic,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых и 1 зеленый...",8
847,logic,"В слове МАТЕМАТИКА стерли 6 букв (возможно, ср...",Перепишем слово в обратном порядке: АКИТАМЕТАМ...,ТЕМА
848,logic,"Восстановив порядок букв в каждой группе, вы п...","Если восстановить данные слова, получим: БАКУ,...",МАРСАКАНД


In [59]:
qaz_df.describe()

,tags,task,solution,answer
count,850,850,850,850
unique,2,833,845,476
top,math,Какой цифрой заканчивается произведение 2*3*5*...,По условию задачи 3 слова посередине имеют бук...,6
freq,554,2,2,29


In [60]:
top = qaz_df['task'].describe()['top']
qaz_df[qaz_df['task'] == top]

,tags,task,solution,answer
7,math,Какой цифрой заканчивается произведение 2*3*5*...,"В произведении есть числа 2 и 5, а как известн...",0
318,math,Какой цифрой заканчивается произведение 2*3*5*...,"Так как в произведении есть числа 2 и 5, а как...",0


In [65]:
qaz_df['task'].loc[7]

'Какой цифрой заканчивается произведение 2*3*5*7*9*11*13*17*19*21*23?'

In [93]:
df = pd.concat([vikhr, qaz_df], ignore_index=True)

In [94]:
df

,task,solution,answer,tags
0,"Небольшое тело, подвешенное на твёрдом стержне...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",2,physic
1,"Небольшое тело, подвешенное на твёрдом стержне...","Вследствие удлинения маятника при нагревании, ...",10.8,physic
2,"Катер пересёк прямую реку шириной 90 м, всё вр...",Катер смещается относительно берега за счёт ск...,6,physic
3,"У Васи есть четыре одинаковых динамометра, оди...",Показания исправных одинаковых динамометров до...,3,physic
4,"Однородный кирпич, имеющий форму прямоугольног...","Пусть длины рёбер кирпича равны a, b и c. Тогд...",3.125,physic
...,...,...,...,...
1139,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых, 1 зеленый ...",7,logic
1140,"В коробке есть три красных, 3 желтых, 1 зелены...","В коробке есть 3 красных, 3 желтых и 1 зеленый...",8,logic
1141,"В слове МАТЕМАТИКА стерли 6 букв (возможно, ср...",Перепишем слово в обратном порядке: АКИТАМЕТАМ...,ТЕМА,logic
1142,"Восстановив порядок букв в каждой группе, вы п...","Если восстановить данные слова, получим: БАКУ,...",МАРСАКАНД,logic


In [95]:
df['tags'].value_counts()

tags
math      751
logic     296
physic     97
Name: count, dtype: int64

In [99]:
df.to_json('data_records.json', orient='records', force_ascii=False, indent=4)

## 

______

## Доп. задачи по физике:

In [4]:
phys = pd.read_json('physic_ex.json')
phys

,problem,answer,solution
0,Первая точка движется вдоль оси Y прямоугольно...,3 м/с,По условию задачи первая точка движется вдоль ...
1,У Ивана есть мерный стаканчик с делениями и гр...,"V = 150 см³, T = 91°C","Пусть теплоемкость воды c, неизвестная масса в..."
2,Лыжные соревнования проходят на круговой трасс...,18 < L < 20 (км),"Пусть длина одного круга трассы равна L, скоро..."
3,В вертикальный цилиндрический сосуд радиусом 1...,"1,8 мм",Плотность шара по условию задачи меньше плотно...
4,Спортсмен-тяжелоатлет поднял штангу массой 200...,800 Дж,Воспользуемся формулой для работы в поле силы ...
5,Открытая с двух концов трубка длиной 76 см до ...,22 см,При вытаскивании трубки с плотно закрытым верх...
6,Тело брошено со скоростью 10 м/с под углом 45°...,5 м,Выберем систему отсчета: начало в точке броска...
7,"Цепочка, составленная из маленьких абсолютно г...","0,6 Н",Время падения последнего звена: t₁ = √(2l/g) ≈...
8,"К плюсу батареи с ЭДС 16,8 В и сопротивлением ...","2,0 В",Резисторы R₁ и R₃ (1 Ом и 2 Ом) соединены посл...
9,"На заряженную частицу, влетающую в однородное ...",1 мкКл,Сила Лоренца: F = q v B sinα. При α = 90°: q =...


In [5]:
phys = phys.rename(columns = {'problem': 'task'})
phys['tags'] = 'physic'
phys

,task,answer,solution,tags
0,Первая точка движется вдоль оси Y прямоугольно...,3 м/с,По условию задачи первая точка движется вдоль ...,physic
1,У Ивана есть мерный стаканчик с делениями и гр...,"V = 150 см³, T = 91°C","Пусть теплоемкость воды c, неизвестная масса в...",physic
2,Лыжные соревнования проходят на круговой трасс...,18 < L < 20 (км),"Пусть длина одного круга трассы равна L, скоро...",physic
3,В вертикальный цилиндрический сосуд радиусом 1...,"1,8 мм",Плотность шара по условию задачи меньше плотно...,physic
4,Спортсмен-тяжелоатлет поднял штангу массой 200...,800 Дж,Воспользуемся формулой для работы в поле силы ...,physic
5,Открытая с двух концов трубка длиной 76 см до ...,22 см,При вытаскивании трубки с плотно закрытым верх...,physic
6,Тело брошено со скоростью 10 м/с под углом 45°...,5 м,Выберем систему отсчета: начало в точке броска...,physic
7,"Цепочка, составленная из маленьких абсолютно г...","0,6 Н",Время падения последнего звена: t₁ = √(2l/g) ≈...,physic
8,"К плюсу батареи с ЭДС 16,8 В и сопротивлением ...","2,0 В",Резисторы R₁ и R₃ (1 Ом и 2 Ом) соединены посл...,physic
9,"На заряженную частицу, влетающую в однородное ...",1 мкКл,Сила Лоренца: F = q v B sinα. При α = 90°: q =...,physic


In [6]:
df1 = pd.read_json('data_records.json')

In [8]:
df_full1 = pd.concat([df1, phys], ignore_index = True)
df_full1

,task,solution,answer,tags
0,"Небольшое тело, подвешенное на твёрдом стержне...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",2,physic
1,"Небольшое тело, подвешенное на твёрдом стержне...","Вследствие удлинения маятника при нагревании, ...",10.8,physic
2,"Катер пересёк прямую реку шириной 90 м, всё вр...",Катер смещается относительно берега за счёт ск...,6,physic
3,"У Васи есть четыре одинаковых динамометра, оди...",Показания исправных одинаковых динамометров до...,3,physic
4,"Однородный кирпич, имеющий форму прямоугольног...","Пусть длины рёбер кирпича равны a, b и c. Тогд...",3.125,physic
...,...,...,...,...
1171,Тело движется в инерциальной системе отсчёта п...,"Изменение импульса: FΔt = Δp = p2 − p1, следов...",15,physic
1172,Пластилиновый шарик массой 200 г и свинцовый б...,Запишем закон сохранения импульса в проекции н...,"0,6 м/с",physic
1173,Температуру разреженного газа увеличил в 2 раз...,Для разреженного газа справедливо уравнение со...,6,physic
1174,Каменный блок лежит на горизонтальной кладке с...,"Давление равно: P = F/S = m*g/S, где F – сила ...","18,5",physic


In [9]:
df_full1['tags'].value_counts()

tags
math      751
logic     296
physic    129
Name: count, dtype: int64

In [10]:
df_full1.to_json('data/ds_part_1.json', orient='records', force_ascii=False, indent=4)

______
## evilfreelancer/MATH-500-Russian

In [11]:
mt500 = load_dataset("evilfreelancer/MATH-500-Russian")

c:\Users\aaron\vkr_tsa\project\vkr_gen_model\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aaron\.cache\huggingface\hub\datasets--evilfreelancer--MATH-500-Russian. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 500/500 [00:00<00:00, 4503.29 examples/s]


In [12]:
mt500

DatasetDict({
    test: Dataset({
        features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id'],
        num_rows: 500
    })
})

In [13]:
mt500 = mt500['test'].to_pandas()
mt500

,problem,solution,answer,subject,level,unique_id
0,"Преобразуйте точку $(0,3)$ из прямоугольных ко...",У нас есть что $r = \sqrt{0^2 + 3^2} = 3.$ Так...,"\left( 3, \frac{\pi}{2} \right)",Precalculus,2,test/precalculus/807.json
1,Определим\n\[p = \sum_{k = 1}^\infty \frac{1}{...,"Мы считаем количество раз, когда $\frac{1}{n^3...",p - q,Intermediate Algebra,5,test/intermediate_algebra/1994.json
2,"Если $f(x) = \frac{3x-2}{x-2}$, чему равно зна...",$f(-2)+f(-1)+f(0)=\frac{3(-2)-2}{-2-2}+\frac{3...,\frac{14}{3},Algebra,3,test/algebra/2584.json
3,Сколько положительных целочисленных делителей ...,Сначала разложим на простые множители $196=2^2...,9,Number Theory,3,test/number_theory/572.json
4,Результаты тренировочного бега команды по крос...,Эвелин прошла большее расстояние за меньшее вр...,Эвелин,Algebra,2,test/algebra/1349.json
...,...,...,...,...,...,...
495,Какова область определения функции $f(x) = \fr...,Внутренний логарифм определен только если $x -...,"(2,12) \cup (12,102)",Intermediate Algebra,4,test/intermediate_algebra/1981.json
496,Пусть $z = 1+i$ и $w = \dfrac{3z+1}{5z+7}$. На...,"Подставляя, получаем $w = \dfrac{3(1+i)+1}{5(1...",\frac{5}{13},Intermediate Algebra,3,test/intermediate_algebra/1232.json
497,Равносторонний восьмиугольник имеет четыре сто...,Восьмиугольник можно разбить на пять квадратов...,\frac{7}{2},Geometry,5,test/geometry/561.json
498,Последовательность $(a_n)$ определена следующи...,"Сначала, если $a_3 = a_1,$ то\n\[a_1 = a_3 = a...",-1,Intermediate Algebra,5,test/intermediate_algebra/1508.json


In [14]:
mt500 = mt500.drop(columns=['unique_id'])

In [15]:
mt500 = mt500.drop_duplicates()

In [16]:
mt500.info()

<class 'pandas.DataFrame'>
Index: 499 entries, 0 to 499
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   problem   499 non-null    str  
 1   solution  499 non-null    str  
 2   answer    499 non-null    str  
 3   subject   499 non-null    str  
 4   level     499 non-null    int64
dtypes: int64(1), str(4)
memory usage: 553.6 KB


In [17]:
mt500['subject'].value_counts()

subject
Algebra                   124
Intermediate Algebra       97
Prealgebra                 82
Number Theory              62
Precalculus                55
Geometry                   41
Counting & Probability     38
Name: count, dtype: int64

In [18]:
mt500['subject'] = 'math'

In [19]:
mt500 = mt500.rename(columns = {'problem': 'task', 'subject': 'tags'})
mt500

,task,solution,answer,tags,level
0,"Преобразуйте точку $(0,3)$ из прямоугольных ко...",У нас есть что $r = \sqrt{0^2 + 3^2} = 3.$ Так...,"\left( 3, \frac{\pi}{2} \right)",math,2
1,Определим\n\[p = \sum_{k = 1}^\infty \frac{1}{...,"Мы считаем количество раз, когда $\frac{1}{n^3...",p - q,math,5
2,"Если $f(x) = \frac{3x-2}{x-2}$, чему равно зна...",$f(-2)+f(-1)+f(0)=\frac{3(-2)-2}{-2-2}+\frac{3...,\frac{14}{3},math,3
3,Сколько положительных целочисленных делителей ...,Сначала разложим на простые множители $196=2^2...,9,math,3
4,Результаты тренировочного бега команды по крос...,Эвелин прошла большее расстояние за меньшее вр...,Эвелин,math,2
...,...,...,...,...,...
495,Какова область определения функции $f(x) = \fr...,Внутренний логарифм определен только если $x -...,"(2,12) \cup (12,102)",math,4
496,Пусть $z = 1+i$ и $w = \dfrac{3z+1}{5z+7}$. На...,"Подставляя, получаем $w = \dfrac{3(1+i)+1}{5(1...",\frac{5}{13},math,3
497,Равносторонний восьмиугольник имеет четыре сто...,Восьмиугольник можно разбить на пять квадратов...,\frac{7}{2},math,5
498,Последовательность $(a_n)$ определена следующи...,"Сначала, если $a_3 = a_1,$ то\n\[a_1 = a_3 = a...",-1,math,5


In [20]:
mt500 = mt500.drop(columns = 'level')
mt500

,task,solution,answer,tags
0,"Преобразуйте точку $(0,3)$ из прямоугольных ко...",У нас есть что $r = \sqrt{0^2 + 3^2} = 3.$ Так...,"\left( 3, \frac{\pi}{2} \right)",math
1,Определим\n\[p = \sum_{k = 1}^\infty \frac{1}{...,"Мы считаем количество раз, когда $\frac{1}{n^3...",p - q,math
2,"Если $f(x) = \frac{3x-2}{x-2}$, чему равно зна...",$f(-2)+f(-1)+f(0)=\frac{3(-2)-2}{-2-2}+\frac{3...,\frac{14}{3},math
3,Сколько положительных целочисленных делителей ...,Сначала разложим на простые множители $196=2^2...,9,math
4,Результаты тренировочного бега команды по крос...,Эвелин прошла большее расстояние за меньшее вр...,Эвелин,math
...,...,...,...,...
495,Какова область определения функции $f(x) = \fr...,Внутренний логарифм определен только если $x -...,"(2,12) \cup (12,102)",math
496,Пусть $z = 1+i$ и $w = \dfrac{3z+1}{5z+7}$. На...,"Подставляя, получаем $w = \dfrac{3(1+i)+1}{5(1...",\frac{5}{13},math
497,Равносторонний восьмиугольник имеет четыре сто...,Восьмиугольник можно разбить на пять квадратов...,\frac{7}{2},math
498,Последовательность $(a_n)$ определена следующи...,"Сначала, если $a_3 = a_1,$ то\n\[a_1 = a_3 = a...",-1,math


In [21]:
df_full2 = pd.concat([df_full1, mt500], ignore_index = True)

In [22]:
df_full2

,task,solution,answer,tags
0,"Небольшое тело, подвешенное на твёрдом стержне...","Перечисленные в условии задачи параметры 𝐿, 𝑚 ...",2,physic
1,"Небольшое тело, подвешенное на твёрдом стержне...","Вследствие удлинения маятника при нагревании, ...",10.8,physic
2,"Катер пересёк прямую реку шириной 90 м, всё вр...",Катер смещается относительно берега за счёт ск...,6,physic
3,"У Васи есть четыре одинаковых динамометра, оди...",Показания исправных одинаковых динамометров до...,3,physic
4,"Однородный кирпич, имеющий форму прямоугольног...","Пусть длины рёбер кирпича равны a, b и c. Тогд...",3.125,physic
...,...,...,...,...
1670,Какова область определения функции $f(x) = \fr...,Внутренний логарифм определен только если $x -...,"(2,12) \cup (12,102)",math
1671,Пусть $z = 1+i$ и $w = \dfrac{3z+1}{5z+7}$. На...,"Подставляя, получаем $w = \dfrac{3(1+i)+1}{5(1...",\frac{5}{13},math
1672,Равносторонний восьмиугольник имеет четыре сто...,Восьмиугольник можно разбить на пять квадратов...,\frac{7}{2},math
1673,Последовательность $(a_n)$ определена следующи...,"Сначала, если $a_3 = a_1,$ то\n\[a_1 = a_3 = a...",-1,math


In [23]:
df_full2.info()

<class 'pandas.DataFrame'>
RangeIndex: 1675 entries, 0 to 1674
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   task      1675 non-null   str  
 1   solution  1675 non-null   str  
 2   answer    1675 non-null   str  
 3   tags      1675 non-null   str  
dtypes: str(4)
memory usage: 1.6 MB


In [25]:
df_full2.describe()

,task,solution,answer,tags
count,1675,1675,1675,1675
unique,1658,1670,861,3
top,Какой цифрой заканчивается произведение 2*3*5*...,По условию задачи 3 слова посередине имеют бук...,6,math
freq,2,2,51,1250


In [26]:
df_full2.to_json('data/ds_part_2.json', orient='records', force_ascii=False, indent=4)